In [ ]:
import re
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from collections import Counter

# Ensure NLTK resources are downloaded
nltk.download('punkt')
nltk.download('stopwords')

# ---------------------------------------------------------
# Paragraph (5-6 sentences about Technology / AI)
# ---------------------------------------------------------
paragraph = (
    "Artificial intelligence is rapidly transforming the world of technology and innovation. "
    "Machine learning algorithms enable computers to learn from massive datasets without explicit programming. "
    "Deep learning neural networks can recognize complex patterns in images, audio, and natural language. "
    "Many modern applications rely on intelligent systems to automate complex repetitive tasks. "
    "Ethical considerations around data privacy and algorithmic bias remain critical topics in AI development. "
    "As technology evolves, artificial intelligence will continue to shape the future of human society."
)

# 1. Lowercase and remove punctuation using re
text_lowered = paragraph.lower()
text_cleaned = re.sub(r'[^\w\s]', '', text_lowered)

print("--- Cleaned Text (Punctuation Removed) ---")
print(text_cleaned, "\n")

# 2. Tokenize text into words and sentences
sentences = sent_tokenize(paragraph)
words_nltk = word_tokenize(text_cleaned)

print(f"Total Sentences: {len(sentences)}")
print(f"Total Words (NLTK Tokenizer): {len(words_nltk)}\n")

# 3. Compare Python split() vs NLTK word_tokenize()
python_split_words = paragraph.lower().split()
nltk_tokenized_words = word_tokenize(paragraph.lower())

print("--- Split vs Word Tokenize Comparison ---")
print("Python split() sample (first 8):", python_split_words[:8])
print("NLTK word_tokenize() sample (first 8):", nltk_tokenized_words[:8], "\n")

# 4. Remove stopwords using NLTK stopwords list
stop_words = set(stopwords.words('english'))
filtered_words = [w for w in words_nltk if w not in stop_words]

# 5. Display Word Frequency Distribution
freq_dist = Counter(filtered_words)
print("--- Word Frequency Distribution (Top 10, Excluding Stopwords) ---")
for word, freq in freq_dist.most_common(10):
    print(f"{word}: {freq}")

In [ ]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# 1. Extract all words with only alphabets using re.findall()
alpha_words = re.findall(r'\b[a-zA-Z]+\b', paragraph.lower())

# 2. Remove stopwords
stop_words = set(stopwords.words('english'))
words_no_stop = [word for word in alpha_words if word not in stop_words]

# 3 & 4. Perform Stemming and Lemmatization
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = [stemmer.stem(word) for word in words_no_stop]
lemmatized_words = [lemmatizer.lemmatize(word) for word in words_no_stop]

# 5. Comparison Table
comparison_df = pd.DataFrame({
    'Original Word': words_no_stop,
    'Porter Stemmer': stemmed_words,
    'WordNet Lemmatizer': lemmatized_words
})

print("--- Stemming vs Lemmatization Output Sample ---")
print(comparison_df.drop_duplicates(subset=['Original Word']).head(12).to_string(index=False))

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 3 Short texts (News / Technology headlines)
texts = [
    "Artificial intelligence transforms diagnostic precision in modern healthcare.",
    "Blockchain networks offer decentralized security for modern financial transactions.",
    "Quantum computing speeds up processing tasks across complex scientific data."
]

# 1. CountVectorizer (Bag of Words)
cv = CountVectorizer(stop_words='english')
bow_matrix = cv.fit_transform(texts)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=cv.get_feature_names_out())

print("--- Bag of Words (BoW) Matrix Sample ---")
print(bow_df.iloc[:, :8])

# 2. TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

# 3. Print and Interpret Top 3 Keywords per Text
feature_names = tfidf.get_feature_names_out()

print("\n--- Top 3 TF-IDF Keywords per Text ---")
for i, text in enumerate(texts):
    row = tfidf_matrix[i].toarray().flatten()
    top_indices = row.argsort()[-3:][::-1]
    top_keywords = [(feature_names[idx], round(row[idx], 4)) for idx in top_indices]

    print(f"\nText {i+1}: \"{text}\"")
    for word, score in top_keywords:
        print(f"  - {word}: {score}")

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 2 Short texts describing different technologies
tech_ai = """
Artificial intelligence enables computers to simulate human intelligence and reasoning.
Machine learning models analyze patterns in data to generate predictions and decisions.
Neural networks excel at recognizing vision, language, and complex sequential inputs.
Modern artificial intelligence drives innovations in robotics, automation, and decision systems.
"""

tech_blockchain = """
Blockchain is a decentralized distributed ledger technology for recording transactions.
It utilizes cryptographic hashing to link data blocks into immutable records.
Smart contracts run automatically on decentralized networks without central intermediaries.
Modern blockchain systems form the foundation of cryptocurrency and Web3 infrastructure.
"""

# Preprocessing helper
def get_word_set(text):
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    return set(words)

# 1. Tokenize into sets
set_ai = get_word_set(tech_ai)
set_blockchain = get_word_set(tech_blockchain)

# 2a. Jaccard Similarity
intersection = set_ai.intersection(set_blockchain)
union = set_ai.union(set_blockchain)
jaccard_sim = len(intersection) / len(union)

# 2b. Cosine Similarity using TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform([tech_ai, tech_blockchain])
cos_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

print(f"Jaccard Similarity : {jaccard_sim:.4f}")
print(f"Cosine Similarity  : {cos_sim:.4f}")
print("Shared Tokens      :", sorted(list(intersection)))

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from textblob import TextBlob

# Sample Product/Service Reviews
reviews = [
    "The laptop boot time is amazingly fast, battery life is outstanding, and display is super crisp!",
    "Customer service was terrible and unhelpful. The product arrived damaged and late.",
    "The package arrived on time. The product functions as expected, nothing special.",
    "Incredible noise-cancelling headphones! Sound quality is clear and battery lasts all day.",
    "Poor build quality. The hinge broke after just two days of light use."
]

classified_reviews = []
positive_texts = []

print("--- Sentiment Analysis Results ---")
for i, review in enumerate(reviews, 1):
    blob = TextBlob(review)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    # Classification logic based on Polarity
    if polarity > 0.1:
        sentiment = "Positive"
        positive_texts.append(review)
    elif polarity < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    print(f"Review {i}: [{sentiment}] | Polarity: {polarity:.2f} | Subjectivity: {subjectivity:.2f}")

# Generate WordCloud for all Positive Reviews
all_positive_text = " ".join(positive_texts)
wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(all_positive_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Positive Product Reviews', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# ~100-word training paragraph
corpus = (
    "Artificial intelligence is transforming every industry by bringing automation and deeper insights. "
    "Machine learning models learn patterns directly from massive volumes of structured and unstructured data. "
    "Neural networks process sequential text and complex images to generate human-like predictions. "
    "As deep learning algorithms advance, intelligent technology will continue driving scientific discovery, "
    "improving decision systems, and shaping the future of global innovation."
)

# 1. Tokenize text using Tokenizer from keras
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])
total_words = len(tokenizer.word_index) + 1

# Create input sequences (N-gram sequences)
input_sequences = []
for sentence in corpus.split('.'):
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Predictors (X) and Label (y)
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# 2. Build simple LSTM model
model = Sequential([
    Embedding(total_words, 32, input_length=max_sequence_len - 1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=150, verbose=0)

# 3. Generate 2-3 new lines / words starting from a seed word
def generate_text(seed_text, next_words, max_seq_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_word_idx = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_idx:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Generate output from seed word
seed_word = "artificial intelligence"
generated_output = generate_text(seed_word, 15, max_sequence_len)
print("--- Generated Text Output ---")
print(generated_output)